# Mixed radix FFT - Float

## Radix-2 SDF Butterfly

In [1]:
import numpy as np
from scipy.fft import fft, ifft

In [2]:
class radix2_PreAdder:
    """
    A class used to represent a hardware preadder of a radix-2 butterfly

    ...

    Attributes
    ----------
    input_a : float
        upper input
    input_b : float
        lower input
    output_add : float
        adder output
    output_sub : float
        subtractor output

    Methods
    -------
    calculate(self)
        Calculates the adder and the subtractor outputs 
    """
    def __init__(self):
        self.input_a = 0.0
        self.input_b = 0.0
        self.output_add = 0.0
        self.output_sub = 0.0


    def calculate(self):
        self.output_add = self.input_a + self.input_b
        self.output_sub = self.input_a - self.input_b
        

class radix2_Rotator:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(2**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        self.half_len = size//2
        N = size
        for i in range(self.half_len):
            k = i * 2**(stage_index)
            self.twiddleROM[i+self.half_len] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE RADIX2, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == self.half_len*2-1):
                self.cnt = 0
            else:
                self.cnt += 1
        

class Fifo:
    full = 0
    cnt = 0

    def __init__(self, depth):
        self.depth = depth
        self.buffer = (np.zeros(depth)).astype(complex)
        # print(self.depth)

    def is_full(self):
        return self.full
    
    def get_output(self):
        return self.buffer[-1]
    
    def shift(self, input_sample):
        self.cnt += 1
        self.buffer = np.roll(self.buffer, 1)
        # print("FIFO input samples = ", input_sample)
        self.buffer[0] = input_sample
        if (self.cnt > self.depth):
            self.full = 1
        else:
            self.full = 0



In [3]:
class radix2_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.size = size
        # self.num_of_stages = num_of_stages
        # self.num_of_samples = 2**(num_of_stages-stage_index)
        self.num_of_samples = size
        self.fifo = Fifo(size//2)
        self.pre_adder = radix2_PreAdder()
        self.rotator = radix2_Rotator(stage_index=stage_index, size=size)

    def isFifoFull(self):
        return self.fifo.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_a = self.fifo.get_output()
        self.pre_adder.input_b = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.size}, add_out = {self.pre_adder.output_add}')
        # print(f'STAGE {self.size}, sub_out = {self.pre_adder.output_sub}')
        
        if (self.op_cnt//(self.num_of_samples/2)): ## other half of the input stream is comming
            self.output_sample = self.pre_adder.output_add
            self.fifo.shift(self.pre_adder.output_sub) 
        else:
            self.output_sample = self.fifo.get_output()
            self.fifo.shift(self.input_sample)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')


In [4]:
stage0 = radix2_SDF_stage(stage_index=0, size=8)
stage1 = radix2_SDF_stage(stage_index=0, size=4)
stage2 = radix2_SDF_stage(stage_index=0, size=2)


# input_vector = [1.0, 2.0, 3.0, 4.0, 0.0, 0.0, 0.0]
# input_vector = [4.0, 3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0]

# input_vector = [8.0, 2.0, 0.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
fft_manual = []
for i in range(len(input_vector)):
    stage0.input_sample = input_vector[i]
    stage0.calculate()
    stage1.input_sample = stage0.output_sample
    stage1.calculate()
    stage2.input_sample = stage1.output_sample
    stage2.calculate()
    print(f'i = {i}, real = {stage2.output_sample.real}, imag = {stage2.output_sample.imag}')
    if i > 6:
        fft_manual.append(stage2.output_sample)


STAGE RADIX2, twiddle = [ 1.00000000e+00+0.j          1.00000000e+00+0.j
  1.00000000e+00+0.j          1.00000000e+00+0.j
  1.00000000e+00+0.j          7.07106781e-01-0.70710678j
  6.12323400e-17-1.j         -7.07106781e-01-0.70710678j]
STAGE RADIX2, twiddle = [1.000000e+00+0.j 1.000000e+00+0.j 1.000000e+00+0.j 6.123234e-17-1.j]
STAGE RADIX2, twiddle = [1.+0.j 1.+0.j]
i = 0, real = 0.0, imag = 0.0
i = 1, real = 0.0, imag = 0.0
i = 2, real = 0.0, imag = 0.0
i = 3, real = 0.0, imag = 0.0
i = 4, real = 0.0, imag = 0.0
i = 5, real = 0.0, imag = 0.0
i = 6, real = 0.0, imag = 0.0
i = 7, real = 36.0, imag = 0.0
i = 8, real = -4.0, imag = 0.0
i = 9, real = -4.0, imag = 4.0
i = 10, real = -3.9999999999999996, imag = -4.0
i = 11, real = -4.0, imag = 9.65685424949238
i = 12, real = -3.9999999999999996, imag = -1.6568542494923797
i = 13, real = -4.0, imag = 1.6568542494923797
i = 14, real = -3.9999999999999987, imag = -9.65685424949238


In [6]:
def bracewell_buneman(xarray, length, log2length):
    ''' 
    bracewell-buneman bit reversal function
    inputs: xarray is array; length is array length; log2length=log2(length).
    output: bit reversed array xarray. 
    '''
    muplus = int((log2length+1)/2)
    mvar = 1
    reverse = np.zeros(length, dtype = int)
    upper_range = muplus+1
    for _ in np.arange(1, upper_range):
        for kvar in np.arange(0, mvar):
            tvar = 2*reverse[kvar]
            reverse[kvar] = tvar
            reverse[kvar+mvar] = tvar+1
        mvar = mvar+mvar
    if (log2length & 0x01):
            mvar = mvar/2

    mvar = int(mvar)
    for qvar in np.arange(1, mvar):
        
        nprime = qvar-mvar
        rprimeprime = reverse[qvar]*mvar
        for pvar in np.arange(0, reverse[qvar]):
            nprime = nprime+mvar
            rprime = rprimeprime+reverse[pvar]
            temp = xarray[nprime]
            xarray[nprime] = xarray[rprime]
            xarray[rprime] = temp
    return xarray

In [7]:
# print(fft_manual)
fft_manual = np.array(bracewell_buneman(fft_manual, len(fft_manual), int(np.log2(len(fft_manual)))))
print(fft_manual)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [8]:
# input_vector = [1.0, 1.0]
input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
# input_vector = [1.0, 2.0, 3.0, 4.0]
fft_numpy = np.fft.fft(input_vector)
print(fft_numpy)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


In [9]:
print(np.allclose(fft_manual,fft_numpy))

True


## Radix-3 SDF Butterfly 

In [34]:
class radix3_PreAdder:
    """
    A class used to represent a hardware preadder of a radix-3 butterfly

    ...

    Attributes
    ----------
    input_0 : float
        first input of preadder
    input_1 : float
        second input of preadder
    input_2 : float
        third input of preadder
    output_0 : float
        first output
    output_1 : float
        second output
    output_2 : float
        thirs output

    Methods
    -------
    calculate(self)
        Calculates the preadder outputs 
    """
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_2
        tmp_2_0 = self.input_1 - self.input_2
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0 + tmp_1_0
        tmp_1_1 = tmp_0_0 - (1/2)*tmp_1_0
        tmp_2_1 = tmp_2_0 * (-1j*np.sqrt(3)/2)
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1
        tmp_1_2 = tmp_1_1 + tmp_2_1
        tmp_2_2 = tmp_1_1 - tmp_2_1
        ###### treci nivo pajplajna ^ (ovo su izlazni registri vrv)
        self.output_0 = tmp_0_2
        self.output_1 = tmp_1_2
        self.output_2 = tmp_2_2

class radix3_Rotator:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(3**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        # self.two_thirds_len = 3**(num_of_stages-stage_index) - (3**(num_of_stages-stage_index)//3)
        self.two_thirds_len = size - size//3
        # print(self.two_thirds_len)
        # N = 3**num_of_stages
        N = size
        if (self.two_thirds_len > 2):
            for i in range(self.two_thirds_len):
                if (i < self.two_thirds_len//2):
                    k = i * 3**(stage_index)
                else:
                    k = 2*(i-self.two_thirds_len//2) * 3**(stage_index)
                # print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.two_thirds_len)] = np.exp(-1j*2*np.pi*k/N)
        
        print(f"STAGE RADIX3, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [37]:
class radix3_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        # self.num_of_stages = num_of_stages
        # self.num_of_samples = 3**(num_of_stages-stage_index)
        self.num_of_samples = size
        # self.fifo_0 = Fifo(3**(num_of_stages-stage_index-1))
        # self.fifo_1 = Fifo(3**(num_of_stages-stage_index-1))
        self.fifo_0 = Fifo(size//3)
        self.fifo_1 = Fifo(size//3)
        self.pre_adder = radix3_PreAdder()
        self.rotator = radix3_Rotator(stage_index=stage_index, size=size)

    def isFifoFull_0(self):
        return self.fifo_0.is_full()
    def isFifoFull_1(self):
        return self.fifo_1.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.input_sample
        self.pre_adder.calculate()

        # print(f'STAGE {self.stage_index}, pre_adder_out_0 = {self.pre_adder.output_0}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_1 = {self.pre_adder.output_1}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_2 = {self.pre_adder.output_2}')
        
        if (self.op_cnt < (self.num_of_samples//3)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//3)) and (self.op_cnt < (self.num_of_samples*2/3))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_1())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [49]:
stage0_radix3 = radix3_SDF_stage(stage_index=0, size=27)
stage0_radix3 = radix3_SDF_stage(stage_index=0, size=9)
stage1_radix3 = radix3_SDF_stage(stage_index=0, size=3)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0]
# input_vector = np.random.random(27)
input_vector_padded = np.append(input_vector, np.zeros(8))
fft_radix3_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix3.input_sample = input_vector_padded[i]
    stage0_radix3.calculate()
    stage1_radix3.input_sample = stage0_radix3.output_sample
    stage1_radix3.calculate()
    # stage2_radix3.input_sample = stage1_radix3.output_sample
    # stage2_radix3.calculate()
    if i >= 8:
        print(f'i = {i}, real = {stage1_radix3.output_sample.real:.4f}, imag = {stage1_radix3.output_sample.imag:.4f}')
        fft_radix3_manual.append(stage1_radix3.output_sample)

fft_radix3_manual = np.array(fft_radix3_manual)

STAGE RADIX3, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.97304487-0.23061587j  0.89363264-0.44879918j
  0.76604444-0.64278761j  0.59715859-0.80212319j  0.39607977-0.91821611j
  0.17364818-0.98480775j -0.05814483-0.99830816j -0.28680323-0.95798951j
  1.        +0.j          0.89363264-0.44879918j  0.59715859-0.80212319j
  0.17364818-0.98480775j -0.28680323-0.95798951j -0.68624164-0.72737364j
 -0.93969262-0.34202014j -0.99323836+0.11609291j -0.83548781+0.54950898j]
STAGE RADIX3, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
  1.        +0.j          0.17364818-0.98480775j -0.93969262-0.34202014j]
STAGE RADIX3, twiddle = [1.+0.j 1.+0.j 1.+0.j]
i = 8, real = 45.0000, imag = 0.0000
i = 9, real = -4.5000,

### Single radix digit inversion

In [13]:
import numpy as np

def digit_reverse_array(arr, radix):
    """
    Perform digit-reversal on a NumPy array for a SINGLE given radix.
    
    Parameters:
        arr (np.ndarray): Input array to be reordered.
        radix (int): The radix (base) for the FFT.
    
    Returns:
        np.ndarray: Reordered array based on digit-reversal indices.
    """
    n = arr.size
    if not np.log(n) / np.log(radix) % 1 == 0:
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        reversed_index = digit_reverse(i, radix, num_digits)
        reordered[reversed_index] = arr[i]
    
    return reordered

def reverse_digit_reverse_array(arr, radix):
    """
    Reverse digit-reversal on a NumPy array for a SINGLE given radix.
    
    Parameters:
        arr (np.ndarray): Input array that was digit-reversed.
        radix (int): The radix (base) used for digit-reversal.
    
    Returns:
        np.ndarray: Array restored to its original order.
    """
    n = arr.size
    if not (np.log(n) / np.log(radix)).is_integer():
        raise ValueError("The size of the array must be a power of the radix.")
    
    num_digits = int(np.log(n) / np.log(radix))
    
    def digit_reverse(index, radix, num_digits):
        reversed_index = 0
        for _ in range(num_digits):
            reversed_index = reversed_index * radix + (index % radix)
            index //= radix
        return reversed_index
    
    reordered = np.empty_like(arr)
    for i in range(n):
        original_index = digit_reverse(i, radix, num_digits)
        reordered[original_index] = arr[i]
    
    return reordered


In [14]:
# print(fft_radix3_manual)
fft_radix3_manual = reverse_digit_reverse_array(fft_radix3_manual, radix=3)
for num in fft_radix3_manual:
    print(num)
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix3_numpy = np.fft.fft(input_vector)
print("\n")
for num in fft_radix3_numpy:
    print(num)
# print(fft_radix3_numpy)

(45+0j)
(-4.5+12.363648387545801j)
(-4.500000000000002+5.362891166673945j)
(-4.5+2.598076211353316j)
(-4.5+0.7934714131880916j)
(-4.499999999999999-0.7934714131880938j)
(-4.5-2.598076211353316j)
(-4.5-5.362891166673945j)
(-4.499999999999999-12.3636483875458j)


(45+0j)
(-4.5+12.363648387545801j)
(-4.499999999999999+5.362891166673945j)
(-4.5+2.598076211353316j)
(-4.499999999999999+0.7934714131880916j)
(-4.5-0.7934714131880929j)
(-4.5-2.598076211353316j)
(-4.500000000000001-5.362891166673945j)
(-4.5-12.363648387545801j)


## Radix-5 SDF Butterfly 

In [15]:
class radix5_PreAdder:
    def __init__(self):
        self.input_0 = 0.0
        self.input_1 = 0.0
        self.input_2 = 0.0
        self.input_3 = 0.0
        self.input_4 = 0.0
        self.output_0 = 0.0
        self.output_1 = 0.0
        self.output_2 = 0.0
        self.output_3 = 0.0
        self.output_4 = 0.0

        self.k1 = -1/4
        self.k2 = 1/2 * (np.cos(2*np.pi/5) - np.cos(4*np.pi/5))
        self.k3 = 1j * (np.sin(4*np.pi/5) - np.sin(2*np.pi/5))
        self.k4 = -1j * np.sin(4*np.pi/5)
        self.k5 = 1j * (np.sin(4*np.pi/5) + np.sin(2*np.pi/5))


    def calculate(self):
        tmp_0_0 = self.input_0
        tmp_1_0 = self.input_1 + self.input_4
        tmp_2_0 = self.input_2 + self.input_3
        tmp_3_0 = self.input_1 - self.input_4
        tmp_4_0 = self.input_2 - self.input_3
        ###### prvi nivo pajplajna ^
        tmp_0_1 = tmp_0_0
        tmp_1_1 = tmp_1_0 + tmp_2_0
        tmp_2_1 = tmp_1_0 - tmp_2_0
        tmp_3_1 = tmp_3_0
        tmp_4_1 = tmp_4_0
        tmp_5_1 = tmp_3_0 + tmp_4_0 # dodatna grana izmedju
        ###### drugi nivo pajplajna ^
        tmp_0_2 = tmp_0_1 + tmp_1_1
        tmp_1_2 = tmp_0_1 + tmp_1_1 * self.k1 #(-0.25)
        # print("tmp_1_2 = ", tmp_1_2.real, " +j ", tmp_1_2.imag)
        tmp_2_2 = tmp_2_1 * self.k2 #0.559
        # print("tmp_2_2 = ", tmp_2_2.real, " +j ", tmp_2_2.imag)
        tmp_3_2 = tmp_3_1 * self.k3 #(-1j*0.363)
        # print("tmp_3_2 = ", tmp_3_2.real, " +j ", tmp_3_2.imag)
        tmp_4_2 = tmp_4_1 * self.k5 #1j*1.539
        # print("tmp_4_2 = ", tmp_4_2.real, " +j ", tmp_4_2.imag)
        tmp_5_2 = tmp_5_1 * self.k4 #(-1j*0.588)
        # print("tmp_5_2 = ", tmp_5_2.real, " +j ", tmp_5_2.imag)
        ###### treci nivo pajplajna ^
        tmp_0_3 = tmp_0_2
        tmp_1_3 = tmp_1_2 + tmp_2_2
        tmp_2_3 = tmp_1_2 - tmp_2_2
        tmp_3_3 = tmp_3_2 + tmp_5_2
        tmp_4_3 = tmp_4_2 + tmp_5_2
        ###### cetvrti nivo pajplajna ^
        self.output_0 = tmp_0_3
        self.output_1 = tmp_1_3 + tmp_3_3
        self.output_2 = tmp_2_3 + tmp_4_3
        self.output_4 = tmp_1_3 - tmp_3_3
        self.output_3 = tmp_2_3 - tmp_4_3

class radix5_Rotator:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input = 0.0
        self.output = 0.0
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(5**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = (np.ones(size)).astype(complex)
        # self.four_fifths_len = 5**(num_of_stages-stage_index) - (5**(num_of_stages-stage_index)//5)
        self.four_fifths_len = size - (size//5)
        N = size
        if (self.four_fifths_len > 4):
            for i in range(self.four_fifths_len):
                if (i < self.four_fifths_len/4):
                    k = i * 5**(stage_index)
                elif ((i >= self.four_fifths_len/4) and (i < self.four_fifths_len/2)):
                    # UPITNO
                    k = 2*(i-self.four_fifths_len//4) * 5**(stage_index)
                elif ((i >= self.four_fifths_len/2) and (i < 3*self.four_fifths_len/4)):
                    # UPITNO
                    k = 3*(i-2*self.four_fifths_len//4) * 5**(stage_index)
                else:
                    k = 4*(i-3*self.four_fifths_len//4) * 5**(stage_index)
                # print("k = ", k)
                self.twiddleROM[i+(len(self.twiddleROM) - self.four_fifths_len)] = np.exp(-1j*2*np.pi*k/N)
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            self.output = self.input * self.twiddleROM[self.cnt]
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [16]:
class radix5_SDF_stage:
    input_sample = 0.0
    output_sample = 0.0
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.num_of_samples = size
        self.fifo_0 = Fifo(size//5)
        self.fifo_1 = Fifo(size//5)
        self.fifo_2 = Fifo(size//5)
        self.fifo_3 = Fifo(size//5)
        self.pre_adder = radix5_PreAdder()
        self.rotator = radix5_Rotator(stage_index=stage_index, size=size)

    def isFifoFull_3(self):
        return self.fifo_3.is_full()
    
    def calculate(self):
        self.pre_adder.input_0 = self.fifo_0.get_output()
        self.pre_adder.input_1 = self.fifo_1.get_output()
        self.pre_adder.input_2 = self.fifo_2.get_output()
        self.pre_adder.input_3 = self.fifo_3.get_output()
        self.pre_adder.input_4 = self.input_sample
        self.pre_adder.calculate()
        
        if (self.op_cnt < (self.num_of_samples//5)): ## other half of the input stream is comming
            self.output_sample = self.fifo_0.get_output()
            self.fifo_0.shift(self.input_sample)
        elif ((self.op_cnt >= (self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*2/5))):
            self.output_sample = self.fifo_1.get_output()
            self.fifo_1.shift(self.input_sample)
        elif ((self.op_cnt >= (2*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*3/5))):
            self.output_sample = self.fifo_2.get_output()
            self.fifo_2.shift(self.input_sample)
        elif ((self.op_cnt >= (3*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*4/5))):
            self.output_sample = self.fifo_3.get_output()
            self.fifo_3.shift(self.input_sample)
        else:
            self.output_sample = self.pre_adder.output_0
            self.fifo_0.shift(self.pre_adder.output_1)
            self.fifo_1.shift(self.pre_adder.output_2)
            self.fifo_2.shift(self.pre_adder.output_3)
            self.fifo_3.shift(self.pre_adder.output_4)

        self.rotator.input = self.output_sample
        self.rotator.rotate(self.isFifoFull_3())
        self.output_sample = self.rotator.output

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [17]:
stage0_radix5 = radix5_SDF_stage(stage_index=0, size=25)
stage1_radix5 = radix5_SDF_stage(stage_index=1, size=5)

input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector = np.arange(25)
# input_vector = np.ones(25)
input_vector_padded = np.append(input_vector, np.zeros(24))
fft_radix5_manual = []
for i in range(len(input_vector_padded)):
    stage0_radix5.input_sample = input_vector_padded[i]
    stage0_radix5.calculate()
    stage1_radix5.input_sample = stage0_radix5.output_sample
    stage1_radix5.calculate()
    
    if i >= 24:
        print("i = ", i, stage1_radix5.output_sample)
        fft_radix5_manual.append(stage1_radix5.output_sample)

fft_radix5_manual = np.array(fft_radix5_manual)

i =  24 (300+0j)
i =  25 (-12.5+17.204774005889668j)
i =  26 (-12.5+4.061496202911331j)
i =  27 (-12.5-4.061496202911331j)
i =  28 (-12.5-17.204774005889668j)
i =  29 (-12.5+98.94768860382283j)
i =  30 (-12.499999999999998+13.311148004059898j)
i =  31 (-12.500000000000002+2.384502527732086j)
i =  32 (-12.500000000000002-5.882053515153152j)
i =  33 (-12.499999999999998-22.737415591013328j)
i =  34 (-12.499999999999995+48.68428568662324j)
i =  35 (-12.499999999999998+10.34089932465595j)
i =  36 (-12.5+0.7864333406706328j)
i =  37 (-12.500000000000004-7.932741219301858j)
i =  38 (-12.500000000000005-31.571396118091315j)
i =  39 (-12.50000000000001+31.57139611809131j)
i =  40 (-12.500000000000005+7.932741219301853j)
i =  41 (-12.5-0.7864333406706292j)
i =  42 (-12.5-10.34089932465595j)
i =  43 (-12.499999999999988-48.68428568662325j)
i =  44 (-12.500000000000004+22.737415591013324j)
i =  45 (-12.49999999999999+5.882053515153132j)
i =  46 (-12.499999999999996-2.3845025277320673j)
i =  47 (-

In [18]:
# print(fft_radix3_manual)

fft_radix5_manual_rev = reverse_digit_reverse_array(fft_radix5_manual, radix=5)

print("Manual fft radix-5")
i = 0
for num in fft_radix5_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1
# fft_radix3_numpy = np.fft.fft([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0])
fft_radix5_numpy = np.fft.fft(input_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_radix5_numpy:
    print(f'{i} | {num:.2f}')
    i+=1
# print(fft_radix3_numpy)

Manual fft radix-5
0 | 300.00+0.00j
1 | -12.50+98.95j
2 | -12.50+48.68j
3 | -12.50+31.57j
4 | -12.50+22.74j
5 | -12.50+17.20j
6 | -12.50+13.31j
7 | -12.50+10.34j
8 | -12.50+7.93j
9 | -12.50+5.88j
10 | -12.50+4.06j
11 | -12.50+2.38j
12 | -12.50+0.79j
13 | -12.50-0.79j
14 | -12.50-2.38j
15 | -12.50-4.06j
16 | -12.50-5.88j
17 | -12.50-7.93j
18 | -12.50-10.34j
19 | -12.50-13.31j
20 | -12.50-17.20j
21 | -12.50-22.74j
22 | -12.50-31.57j
23 | -12.50-48.68j
24 | -12.50-98.95j


Numpy fft
0 | 300.00+0.00j
1 | -12.50+98.95j
2 | -12.50+48.68j
3 | -12.50+31.57j
4 | -12.50+22.74j
5 | -12.50+17.20j
6 | -12.50+13.31j
7 | -12.50+10.34j
8 | -12.50+7.93j
9 | -12.50+5.88j
10 | -12.50+4.06j
11 | -12.50+2.38j
12 | -12.50+0.79j
13 | -12.50-0.79j
14 | -12.50-2.38j
15 | -12.50-4.06j
16 | -12.50-5.88j
17 | -12.50-7.93j
18 | -12.50-10.34j
19 | -12.50-13.31j
20 | -12.50-17.20j
21 | -12.50-22.74j
22 | -12.50-31.57j
23 | -12.50-48.68j
24 | -12.50-98.95j


In [19]:
digit_reversed_data = np.array([0, 5, 10, 15, 20, 1, 6, 11, 16, 21, 2, 7, 12, 17, 22, 3, 8, 13, 18, 23, 4, 9, 14, 19, 24])
radix = 5

# Perform reverse digit-reversal
original_data = reverse_digit_reverse_array(digit_reversed_data, radix)
print("Digit-Reversed Data:", digit_reversed_data)
print("Restored Original Data:", original_data)

Digit-Reversed Data: [ 0  5 10 15 20  1  6 11 16 21  2  7 12 17 22  3  8 13 18 23  4  9 14 19
 24]
Restored Original Data: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24]


## Digit inversion

In [20]:
def digit_reverse(radices):
    radices_rev = np.copy(radices)
    radices_rev = radices_rev[::-1]

    mr_fft_len = np.prod(radices)
    indices = np.arange(mr_fft_len)

    mult_factors = []
    tmp_len = mr_fft_len
    for radix in radices:
        tmp_len = tmp_len//radix
        mult_factors.append(tmp_len)
    
    mult_factors_rev = []
    tmp_len = mr_fft_len
    for radix in radices_rev:
        tmp_len = tmp_len//radix
        mult_factors_rev.append(tmp_len)
    

    decomp = []
    for index in indices:
        tmp_decomposition = []
        tmp_index = index
        for mult_factor in mult_factors:
            tmp_decomposition.append(tmp_index // mult_factor)
            tmp_index = tmp_index % mult_factor
        decomp.append(tmp_decomposition)
    
    decomp = np.array(decomp)
    decomp = decomp.T[::-1]

    indices_rev = np.multiply(np.tile(mult_factors_rev, (mr_fft_len,1)), decomp.T).sum(axis=1)

    return indices_rev

## Decompositon of N on $2^i \cdot 3^j \cdot 5^k$

The max number of subcarriers in OFDM (5G NR standard) is 3300 $(275 * 12)$, so besides radix 2, 3 and 5, a radix 11 butterfly ($275 = 5^2 \cdot 11^1$) is also needed to achieve the best performance (lowest spectral leakage).
In the next few cells radix powers and the list of all possible FFT sizes will be generated.

It is also possible to avoid the radix 11 butterfly usage if the system is willing to introduce some error because of the spectral leakage. In that case, radices 2, 3 and 5 could generate FFT of size 3840.

In [21]:
i_arr = []
j_arr = []
k_arr = []
for i in range(10):
    for j in range(10):
        for k in range(10):
            if ((((2**i) * (3**j) * (5**k)) <= 275)):
                i_arr.append(i)
                j_arr.append(j)
                k_arr.append(k)

In [26]:
N_arr = []
i_arr = np.unique(i_arr)
j_arr = np.unique(j_arr)
k_arr = np.unique(k_arr)
stage_nums = []

for i in i_arr:
    for j in j_arr:
        for k in k_arr:
            if((12 * ((2**i) * (3**j) * (5**k))) <= 3300):
            # if (not ((12 * (2**i * 3**j * 5**k)) in N_arr)):
                # print(f"i = {i}, j = {j}, k = {k}")
                N_arr.append(12 * (2**i * 3**j * 5**k))
                stage_nums.append(i+2 + j+1 + k)
                print("2: ", i+2, " | 3: ", j+1, " | 5: ", k)
                print(12 * (2**i * 3**j * 5**k))

N_arr = np.array(N_arr)
N_arr.sort()

print("Largest mixed-radix FFT sequence length = ", max(N_arr))
print("Number of radix combinations = ", len(N_arr))

print("Sequence lengths = ", N_arr)
print("Number of radix-2 stages = ", i_arr+2)
print("Number of radix-3 stages = ", j_arr+1)
print("Number of radix-5 stages = ", k_arr)
print("stage nums = ", np.array(stage_nums))


2:  2  | 3:  1  | 5:  0
12
2:  2  | 3:  1  | 5:  1
60
2:  2  | 3:  1  | 5:  2
300
2:  2  | 3:  1  | 5:  3
1500
2:  2  | 3:  2  | 5:  0
36
2:  2  | 3:  2  | 5:  1
180
2:  2  | 3:  2  | 5:  2
900
2:  2  | 3:  3  | 5:  0
108
2:  2  | 3:  3  | 5:  1
540
2:  2  | 3:  3  | 5:  2
2700
2:  2  | 3:  4  | 5:  0
324
2:  2  | 3:  4  | 5:  1
1620
2:  2  | 3:  5  | 5:  0
972
2:  2  | 3:  6  | 5:  0
2916
2:  3  | 3:  1  | 5:  0
24
2:  3  | 3:  1  | 5:  1
120
2:  3  | 3:  1  | 5:  2
600
2:  3  | 3:  1  | 5:  3
3000
2:  3  | 3:  2  | 5:  0
72
2:  3  | 3:  2  | 5:  1
360
2:  3  | 3:  2  | 5:  2
1800
2:  3  | 3:  3  | 5:  0
216
2:  3  | 3:  3  | 5:  1
1080
2:  3  | 3:  4  | 5:  0
648
2:  3  | 3:  4  | 5:  1
3240
2:  3  | 3:  5  | 5:  0
1944
2:  4  | 3:  1  | 5:  0
48
2:  4  | 3:  1  | 5:  1
240
2:  4  | 3:  1  | 5:  2
1200
2:  4  | 3:  2  | 5:  0
144
2:  4  | 3:  2  | 5:  1
720
2:  4  | 3:  3  | 5:  0
432
2:  4  | 3:  3  | 5:  1
2160
2:  4  | 3:  4  | 5:  0
1296
2:  5  | 3:  1  | 5:  0
96
2:  5  | 3:  1 

# Combining the stages with different radices

## Radix-3 and radix-2

In [52]:
N = 18
test_vector = np.random.random(N)

fft_stage0 = radix3_SDF_stage(stage_index=0, size=18)
fft_stage1 = radix3_SDF_stage(stage_index=0, size=6)
fft_stage2 = radix2_SDF_stage(stage_index=0, size=2)

test_vector_padded = np.append(test_vector, np.zeros(17))
fft_mixed_radix_manual = []
for i in range(len(test_vector_padded)):
    fft_stage0.input_sample = test_vector_padded[i]
    fft_stage0.calculate()
    fft_stage1.input_sample = fft_stage0.output_sample
    fft_stage1.calculate()
    fft_stage2.input_sample = fft_stage1.output_sample
    fft_stage2.calculate()
    print(fft_stage2.output_sample)
    if i >= 17:
        fft_mixed_radix_manual.append(fft_stage2.output_sample)

fft_mixed_radix_manual = np.array(fft_mixed_radix_manual)

radices = [3,3,2]
rev_seq = digit_reverse(radices)

fft_mixed_radix_manual_rev = np.zeros(N).astype(complex)
i = 0
while (i < N):
    fft_mixed_radix_manual_rev[rev_seq[i]] = fft_mixed_radix_manual[i]
    i += 1

STAGE RADIX3, twiddle = [ 1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          1.        +0.j          1.        +0.j
  1.        +0.j          0.93969262-0.34202014j  0.76604444-0.64278761j
  0.5       -0.8660254j   0.17364818-0.98480775j -0.17364818-0.98480775j
  1.        +0.j          0.76604444-0.64278761j  0.17364818-0.98480775j
 -0.5       -0.8660254j  -0.93969262-0.34202014j -0.93969262+0.34202014j]
STAGE RADIX3, twiddle = [ 1. +0.j         1. +0.j         1. +0.j         0.5-0.8660254j
  1. +0.j        -0.5-0.8660254j]
STAGE RADIX2, twiddle = [1.+0.j 1.+0.j]
0.0
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
0j
(9.258690656125811+0j)
(-1.5501247627390744+0j)
(0.36234193730177167+1.4658106645074382j)
(1.812674663856583-0.002840318850158785j)
(1.8126746638565825+0.002840318850158896j)
(0.362341937301772-1.4658106645074385j)
(-1.0287751237059277+0.5720224552806883j)
(0.19642129139410558+0.06527646400672465j)
(-0.9932220557079776-0.471186130146301

In [48]:
print("Manual fft mixed radix")
i = 0
for num in fft_mixed_radix_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1

fft_mixed_radix_numpy = np.fft.fft(test_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_mixed_radix_numpy:
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | 6.37+0.00j
1 | 0.76+1.47j
2 | 0.42+0.22j
3 | 0.22+0.26j
4 | -1.13-0.02j
5 | 0.21-0.68j
6 | -0.11-0.12j
7 | -0.32-0.44j
8 | -0.39-0.75j
9 | -0.95+0.00j
10 | -0.39+0.75j
11 | -0.32+0.44j
12 | -0.11+0.12j
13 | 0.21+0.68j
14 | -1.13+0.02j
15 | 0.22-0.26j
16 | 0.42-0.22j
17 | 0.76-1.47j


Numpy fft
0 | 6.37+0.00j
1 | 0.76+1.47j
2 | 0.42+0.22j
3 | 0.22+0.26j
4 | -1.13-0.02j
5 | 0.21-0.68j
6 | -0.11-0.12j
7 | -0.32-0.44j
8 | -0.39-0.75j
9 | -0.95-0.00j
10 | -0.39+0.75j
11 | -0.32+0.44j
12 | -0.11+0.12j
13 | 0.21+0.68j
14 | -1.13+0.02j
15 | 0.22-0.26j
16 | 0.42-0.22j
17 | 0.76-1.47j


## Radix-5 and radix-2

In [29]:
N = 10
test_vector = np.random.random(N)

radices = [5,2]
rev_seq = digit_reverse(radices)

fft_stage0 = radix5_SDF_stage(stage_index=0, size=10)
fft_stage1 = radix2_SDF_stage(stage_index=0, size=2)

test_vector_padded = np.append(test_vector, np.zeros(9))
fft_mixed_radix_manual = []
for i in range(len(test_vector_padded)):
    fft_stage0.input_sample = test_vector_padded[i]
    fft_stage0.calculate()
    fft_stage1.input_sample = fft_stage0.output_sample
    fft_stage1.calculate()
    print(fft_stage1.output_sample)
    if i >= 9:
        fft_mixed_radix_manual.append(fft_stage1.output_sample)

fft_mixed_radix_manual = np.array(fft_mixed_radix_manual)

fft_mixed_radix_manual_rev = np.zeros(N).astype(complex)
i = 0
while (i < N):
    fft_mixed_radix_manual_rev[rev_seq[i]] = fft_mixed_radix_manual[i]
    i += 1

0.0
0j
0j
0j
0j
0j
0j
0j
0j
(5.060643973711516+0j)
(-0.6875991624223237+0j)
(1.3211102382978739-0.2129836942977703j)
(-0.1267990454067638-0.1419534689498303j)
(0.5104582414589625-0.6726080944705547j)
(-1.0982337931546806-0.3507135917934804j)
(-1.0982337931546806+0.3507135917934803j)
(0.5104582414589626+0.6726080944705548j)
(-0.1267990454067638+0.14195346894983019j)
(1.3211102382978739+0.21298369429777042j)


In [30]:
print("Manual fft mixed radix")
# fft_mixed_radix_manual = fft_mixed_radix_manual[rev_seq]
i = 0
i = 0
for num in fft_mixed_radix_manual_rev:
    print(f'{i} | {num:.2f}')
    i+=1

fft_mixed_radix_numpy = np.fft.fft(test_vector)
print("\n")

print("Numpy fft")
i=0
for num in fft_mixed_radix_numpy:
    print(f'{i} | {num:.2f}')
    i+=1

Manual fft mixed radix
0 | 5.06+0.00j
1 | 1.32-0.21j
2 | 0.51-0.67j
3 | -1.10+0.35j
4 | -0.13+0.14j
5 | -0.69+0.00j
6 | -0.13-0.14j
7 | -1.10-0.35j
8 | 0.51+0.67j
9 | 1.32+0.21j


Numpy fft
0 | 5.06+0.00j
1 | 1.32-0.21j
2 | 0.51-0.67j
3 | -1.10+0.35j
4 | -0.13+0.14j
5 | -0.69-0.00j
6 | -0.13-0.14j
7 | -1.10-0.35j
8 | 0.51+0.67j
9 | 1.32+0.21j


# Mixed radix FFT - FXP

In [26]:
import numpy as np
from fxpmath import Fxp
from scipy.fft import fft, ifft

## Data fixed-point format

In [27]:
# type of fixed-point numbers used
DATA = Fxp(None, True, dtype='fxp-s32/12')

## Radix-2 SDF Butterfly FXP

In [28]:
class radix2_PreAdder_fxp:
    """
    A class used to represent a fixed-point hardware preadder of a radix-2 butterfly

    ...

    Attributes
    ----------
    input_a : fxp
        upper input
    input_b : fxp
        lower input
    output_add : fxp
        adder output
    output_sub : fxp
        subtractor output

    Methods
    -------
    calculate(self)
        Calculates the adder and the subtractor outputs 
    """
    def __init__(self):
        self.input_a_r = Fxp(0.0).like(DATA)
        self.input_a_i = Fxp(0.0).like(DATA)

        self.input_b_r = Fxp(0.0).like(DATA)
        self.input_b_i = Fxp(0.0).like(DATA)

        self.output_add_r = Fxp(0.0).like(DATA)
        self.output_add_i = Fxp(0.0).like(DATA)

        self.output_sub_r = Fxp(0.0).like(DATA)
        self.output_sub_i = Fxp(0.0).like(DATA)


    def calculate(self):
        # print("Preadder input a = ", self.input_a_r, self.input_a_i)
        # print("Preadder input b = ", self.input_b_r, self.input_b_i)
        self.output_add_r = self.input_a_r + self.input_b_r
        self.output_add_i = self.input_a_i + self.input_b_i
        self.output_sub_r = self.input_a_r - self.input_b_r
        self.output_sub_i = self.input_a_i - self.input_b_i
        

class radix2_Rotator_fxp:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input_r = Fxp(0.0).like(DATA)
        self.input_i = Fxp(0.0).like(DATA)
        self.output_r = Fxp(0.0).like(DATA)
        self.output_i = Fxp(0.0).like(DATA)
        self.stage_index = stage_index
        # self.twiddleROM = (np.ones(2**(num_of_stages-stage_index))).astype(complex)
        np_rom = np.zeros((size,2))
        np_rom[:,0].fill(1.0)
        
        self.twiddleROM = Fxp(np_rom).like(DATA)
        self.half_len = size//2
        N = size
        for i in range(self.half_len):
            k = i * 2**(stage_index)
            twiddle_factor = np.exp(-1j*2*np.pi*k/N)

            self.twiddleROM[i+self.half_len, 0] = twiddle_factor.real
            self.twiddleROM[i+self.half_len, 1] = twiddle_factor.imag
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        x_r = Fxp(0.0).like(DATA)
        x_i = Fxp(0.0).like(DATA)
        y_r = Fxp(0.0).like(DATA)
        y_i = Fxp(0.0).like(DATA)
        z_r = Fxp(0.0).like(DATA)
        z_i = Fxp(0.0).like(DATA)
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            x_r(self.input_r)
            x_i(self.input_i)
            y_r(self.twiddleROM[self.cnt, 0])
            y_i(self.twiddleROM[self.cnt, 1])
            # self.output = self.input * self.twiddleROM[self.cnt]
            z_r.set_val(x_r*y_r - x_i*y_i)
            z_i.set_val(x_r*y_i + x_i*y_r)

            self.output_r.set_val(z_r)
            self.output_i.set_val(z_i)
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == self.half_len*2-1):
                self.cnt = 0
            else:
                self.cnt += 1
        

class Fifo_fxp:
    full = 0
    cnt = 0

    def __init__(self, depth):
        self.depth = depth
        self.buffer = Fxp(np.zeros((depth,2))).like(DATA)
        # print(self.depth)

    def is_full(self):
        return self.full
    
    def get_output(self):
        return self.buffer[-1]
    
    def shift(self, input_sample_r, input_sample_i):
        self.cnt += 1
        # print("FIFO buffer before roll = ", self.buffer)
        self.buffer = np.roll(self.buffer, 1, axis=0).like(DATA)
        # print("FIFO input samples = ", input_sample_r, input_sample_i)
        self.buffer[0,0] = input_sample_r
        self.buffer[0,1] = input_sample_i
        # print("FIFO buffer after roll = ", self.buffer)
        if (self.cnt > self.depth):
            self.full = 1
        else:
            self.full = 0



In [29]:
class radix2_SDF_fxp_stage:
    input_sample_r = Fxp(0.0).like(DATA)
    input_sample_i = Fxp(0.0).like(DATA)
    output_sample_r = Fxp(0.0).like(DATA)
    output_sample_i = Fxp(0.0).like(DATA)
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.size = size
        # self.num_of_stages = num_of_stages
        # self.num_of_samples = 2**(num_of_stages-stage_index)
        self.num_of_samples = size
        self.fifo = Fifo_fxp(size//2)
        self.pre_adder = radix2_PreAdder_fxp()
        self.rotator = radix2_Rotator_fxp(stage_index=stage_index, size=size)

    def isFifoFull(self):
        return self.fifo.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        fifo_out_reg = self.fifo.get_output()
        # print("AAA = ", fifo_out_reg)
        self.pre_adder.input_a_r.set_val(fifo_out_reg[0])
        self.pre_adder.input_a_i.set_val(fifo_out_reg[1])
        self.pre_adder.input_b_r.set_val(self.input_sample_r)
        self.pre_adder.input_b_i.set_val(self.input_sample_i)
        self.pre_adder.calculate()

        # print(f'STAGE {self.size}, add_out r = {self.pre_adder.output_add_r} i = {self.pre_adder.output_add_i}')
        # print(f'STAGE {self.size}, sub_out r = {self.pre_adder.output_sub_r} i = {self.pre_adder.output_sub_i}')
        # print(f'STAGE {self.size}, sub_out = {self.pre_adder.output_sub}')
        
        if (self.op_cnt//(self.num_of_samples/2)): ## other half of the input stream is comming
            self.output_sample_r.set_val(self.pre_adder.output_add_r)
            self.output_sample_i.set_val(self.pre_adder.output_add_i)
            self.fifo.shift(self.pre_adder.output_sub_r, self.pre_adder.output_sub_i) 
        else:
            fifo_out_reg = self.fifo.get_output()
            self.output_sample_r.set_val(fifo_out_reg[0])
            self.output_sample_i.set_val(fifo_out_reg[1])
            self.fifo.shift(self.input_sample_r, self.input_sample_i)

        self.rotator.input_r.set_val(self.output_sample_r)
        self.rotator.input_i.set_val(self.output_sample_i)
        self.rotator.rotate(self.isFifoFull())
        self.output_sample_r.set_val(self.rotator.output_r)
        self.output_sample_i.set_val(self.rotator.output_i)

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')


In [30]:
stage0 = radix2_SDF_fxp_stage(stage_index=0, size=8)
stage1 = radix2_SDF_fxp_stage(stage_index=0, size=4)
stage2 = radix2_SDF_fxp_stage(stage_index=0, size=2)


# input_vector = [1.0, 2.0, 3.0, 4.0, 0.0, 0.0, 0.0]
# input_vector = [4.0, 3.0, 2.0, 1.0, 0.0, 0.0, 0.0, 0.0]

# input_vector = [1.0, 1.0, 0.0]
input_vector_r = Fxp([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]).like(DATA)
input_vector_i = Fxp(np.zeros(15)).like(DATA)
fft_manual_fxp = []
for i in range(len(input_vector_r)):
    stage0.input_sample_r.set_val(input_vector_r[i])
    stage0.input_sample_i.set_val(input_vector_i[i])
    stage0.calculate()
    stage1.input_sample_r.set_val(stage0.output_sample_r)
    stage1.input_sample_i.set_val(stage0.output_sample_i)
    stage1.calculate()
    stage2.input_sample_r.set_val(stage1.output_sample_r)
    stage2.input_sample_i.set_val(stage1.output_sample_i)
    stage2.calculate()
    print(f'i = {i}, real = {stage2.output_sample_r}, imag = {stage2.output_sample_i}')
    if i > 6:
        fft_manual_fxp.append(complex(stage2.output_sample_r, stage2.output_sample_i))

fft_manual_fxp = np.array(fft_manual_fxp)

i = 0, real = 0.0, imag = 0.0
i = 1, real = 0.0, imag = 0.0
i = 2, real = 0.0, imag = 0.0
i = 3, real = 0.0, imag = 0.0
i = 4, real = 0.0, imag = 0.0
i = 5, real = 0.0, imag = 0.0
i = 6, real = 0.0, imag = 0.0
i = 7, real = 36.0, imag = 0.0
i = 8, real = -4.0, imag = 0.0
i = 9, real = -4.0, imag = 4.0
i = 10, real = -4.0, imag = -4.0
i = 11, real = -4.0, imag = 9.65625
i = 12, real = -4.0, imag = -1.65625
i = 13, real = -4.0, imag = 1.65625
i = 14, real = -4.0, imag = -9.65625


In [31]:
fft_manual_fxp = np.array(bracewell_buneman(fft_manual_fxp, len(fft_manual_fxp), int(np.log2(len(fft_manual_fxp)))))
print(fft_manual_fxp)

[36.+0.j      -4.+9.65625j -4.+4.j      -4.+1.65625j -4.+0.j
 -4.-1.65625j -4.-4.j      -4.-9.65625j]


In [32]:
input_vector_float = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0]
fft_numpy = np.fft.fft(input_vector_float)
print(fft_numpy)

[36.+0.j         -4.+9.65685425j -4.+4.j         -4.+1.65685425j
 -4.+0.j         -4.-1.65685425j -4.-4.j         -4.-9.65685425j]


## Radix-3 SDF Butterfly FXP

In [33]:
class radix3_PreAdder_fxp:
    """
    A class used to represent a hardware preadder of a radix-3 butterfly

    ...

    Attributes
    ----------
    input_0 : fxp
        first input of preadder
    input_1 : fxp
        second input of preadder
    input_2 : fxp
        third input of preadder
    output_0 : fxp
        first output
    output_1 : fxp
        second output
    output_2 : fxp
        thirs output

    Methods
    -------
    calculate(self)
        Calculates the preadder outputs 
    """
    def __init__(self):
        self.input_0_r = Fxp(0.0).like(DATA)
        self.input_0_i = Fxp(0.0).like(DATA)

        self.input_1_r = Fxp(0.0).like(DATA)
        self.input_1_i = Fxp(0.0).like(DATA)

        self.input_2_r = Fxp(0.0).like(DATA)
        self.input_2_i = Fxp(0.0).like(DATA)

        self.output_0_r = Fxp(0.0).like(DATA)
        self.output_0_i = Fxp(0.0).like(DATA)

        self.output_1_r = Fxp(0.0).like(DATA)
        self.output_1_i = Fxp(0.0).like(DATA)
        
        self.output_2_r = Fxp(0.0).like(DATA)
        self.output_2_i = Fxp(0.0).like(DATA)

        self.mult_const_cmplx = (-1j*np.sqrt(3)/2)
        self.mult_const_r = Fxp(self.mult_const_cmplx.real).like(DATA)
        self.mult_const_i = Fxp(self.mult_const_cmplx.imag).like(DATA)


    def calculate(self):
        tmp_0_0_r = self.input_0_r
        tmp_0_0_i = self.input_0_i

        tmp_1_0_r = self.input_1_r + self.input_2_r
        tmp_1_0_i = self.input_1_i + self.input_2_i

        tmp_2_0_r = self.input_1_r - self.input_2_r
        tmp_2_0_i = self.input_1_i - self.input_2_i
        ###### prvi nivo pajplajna ^
        tmp_0_1_r = tmp_0_0_r + tmp_1_0_r
        tmp_0_1_i = tmp_0_0_i + tmp_1_0_i

        tmp_1_1_r = tmp_0_0_r - (1/2)*tmp_1_0_r
        tmp_1_1_i = tmp_0_0_i - (1/2)*tmp_1_0_i

        ### Kompleksno mnozenje kompleksnom konstantom
        tmp_2_1_r = tmp_2_0_r * self.mult_const_r - tmp_2_0_i * self.mult_const_i
        tmp_2_1_i = tmp_2_0_r * self.mult_const_i + tmp_2_0_i * self.mult_const_r
        ###### drugi nivo pajplajna ^
        tmp_0_2_r = tmp_0_1_r
        tmp_0_2_i = tmp_0_1_i

        tmp_1_2_r = tmp_1_1_r + tmp_2_1_r
        tmp_1_2_i = tmp_1_1_i + tmp_2_1_i

        tmp_2_2_r = tmp_1_1_r - tmp_2_1_r
        tmp_2_2_i = tmp_1_1_i - tmp_2_1_i
        ###### treci nivo pajplajna ^ (ovo su izlazni registri vrv)
        self.output_0_r = tmp_0_2_r
        self.output_0_i = tmp_0_2_i

        self.output_1_r = tmp_1_2_r
        self.output_1_i = tmp_1_2_i

        self.output_2_r = tmp_2_2_r
        self.output_2_i = tmp_2_2_i


class radix3_Rotator_fxp:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input_r = Fxp(0.0).like(DATA)
        self.input_i = Fxp(0.0).like(DATA)

        self.output_r = Fxp(0.0).like(DATA)
        self.output_i = Fxp(0.0).like(DATA)
        self.stage_index = stage_index

        np_rom = np.zeros((size,2))
        np_rom[:,0].fill(1.0)
        # self.twiddleROM = (np.ones(3**(num_of_stages-stage_index))).astype(complex)
        self.twiddleROM = Fxp(np_rom).like(DATA)
        # self.two_thirds_len = 3**(num_of_stages-stage_index) - (3**(num_of_stages-stage_index)//3)
        self.two_thirds_len = size - size//3
        # print(self.two_thirds_len)
        # N = 3**num_of_stages
        N = size
        if (self.two_thirds_len > 2):
            for i in range(self.two_thirds_len):
                if (i < self.two_thirds_len//2):
                    k = i * 3**(stage_index)
                else:
                    k = 2*(i-self.two_thirds_len//2) * 3**(stage_index)
                # print("k = ", k)
                twiddle_factor = np.exp(-1j*2*np.pi*k/N)
                self.twiddleROM[i+(len(self.twiddleROM) - self.two_thirds_len), 0] = twiddle_factor.real
                self.twiddleROM[i+(len(self.twiddleROM) - self.two_thirds_len), 1] = twiddle_factor.imag
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        x_r = Fxp(0.0).like(DATA)
        x_i = Fxp(0.0).like(DATA)
        y_r = Fxp(0.0).like(DATA)
        y_i = Fxp(0.0).like(DATA)
        z_r = Fxp(0.0).like(DATA)
        z_i = Fxp(0.0).like(DATA)
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            x_r(self.input_r)
            x_i(self.input_i)
            y_r(self.twiddleROM[self.cnt, 0])
            y_i(self.twiddleROM[self.cnt, 1])
            # self.output = self.input * self.twiddleROM[self.cnt]
            z_r.set_val(x_r*y_r - x_i*y_i)
            z_i.set_val(x_r*y_i + x_i*y_r)

            self.output_r.set_val(z_r)
            self.output_i.set_val(z_i)
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [34]:
class radix3_SDF_fxp_stage:
    input_sample_r = Fxp(0.0).like(DATA)
    input_sample_i = Fxp(0.0).like(DATA)
    output_sample_r = Fxp(0.0).like(DATA)
    output_sample_i = Fxp(0.0).like(DATA)
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.num_of_samples = size
        self.fifo_0 = Fifo_fxp(size//3)
        self.fifo_1 = Fifo_fxp(size//3)
        self.pre_adder = radix3_PreAdder_fxp()
        self.rotator = radix3_Rotator_fxp(stage_index=stage_index, size=size)

    def isFifoFull_0(self):
        return self.fifo_0.is_full()
    def isFifoFull_1(self):
        return self.fifo_1.is_full()
    
    def calculate(self):
        # print(f'STAGE {self.stage_index}, op_cnt = ', self.op_cnt)
        fifo_0_out_reg = self.fifo_0.get_output()
        fifo_1_out_reg = self.fifo_1.get_output()

        self.pre_adder.input_0_r.set_val(fifo_0_out_reg[0])
        self.pre_adder.input_0_i.set_val(fifo_0_out_reg[1])

        self.pre_adder.input_1_r.set_val(fifo_1_out_reg[0])
        self.pre_adder.input_1_i.set_val(fifo_1_out_reg[1])

        self.pre_adder.input_2_r = self.input_sample_r
        self.pre_adder.input_2_i = self.input_sample_i

        self.pre_adder.calculate()

        # print(f'STAGE {self.stage_index}, pre_adder_out_0 = {self.pre_adder.output_0}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_1 = {self.pre_adder.output_1}')
        # print(f'STAGE {self.stage_index}, pre_adder_out_2 = {self.pre_adder.output_2}')
        
        if (self.op_cnt < (self.num_of_samples//3)): ## other half of the input stream is comming
            self.output_sample_r.set_val(fifo_0_out_reg[0])
            self.output_sample_i.set_val(fifo_0_out_reg[1])

            self.fifo_0.shift(self.input_sample_r, self.input_sample_i)
        elif ((self.op_cnt >= (self.num_of_samples//3)) and (self.op_cnt < (self.num_of_samples*2/3))):
            self.output_sample_r.set_val(fifo_1_out_reg[0])
            self.output_sample_i.set_val(fifo_1_out_reg[1])
            self.fifo_1.shift(self.input_sample_r, self.input_sample_i)
        else:
            self.output_sample_r.set_val(self.pre_adder.output_0_r)
            self.output_sample_i.set_val(self.pre_adder.output_0_i)

            self.fifo_0.shift(self.pre_adder.output_1_r, self.pre_adder.output_1_i)
            self.fifo_1.shift(self.pre_adder.output_2_r, self.pre_adder.output_2_i)

        self.rotator.input_r.set_val(self.output_sample_r)
        self.rotator.input_i.set_val(self.output_sample_i)

        self.rotator.rotate(self.isFifoFull_1())

        self.output_sample_r = self.rotator.output_r
        self.output_sample_i = self.rotator.output_i

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [35]:
# stage0_radix3_fxp = radix3_SDF_fxp_stage(stage_index=0, size=27)
stage0_radix3_fxp = radix3_SDF_fxp_stage(stage_index=1, size=9)
stage1_radix3_fxp = radix3_SDF_fxp_stage(stage_index=1, size=3)

input_vector_r = Fxp([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]).like(DATA)
input_vector_i = Fxp(np.zeros(17)).like(DATA)

fft_radix3_manual_fxp = []
for i in range(len(input_vector_r)):
    stage0_radix3_fxp.input_sample_r = input_vector_r[i]
    stage0_radix3_fxp.input_sample_i = input_vector_i[i]
    stage0_radix3_fxp.calculate()
    stage1_radix3_fxp.input_sample_r = stage0_radix3_fxp.output_sample_r
    stage1_radix3_fxp.input_sample_i = stage0_radix3_fxp.output_sample_i
    stage1_radix3_fxp.calculate()
    if i > 7:
        print(f'i = {i}, real = {stage1_radix3_fxp.output_sample_r}, imag = {stage1_radix3_fxp.output_sample_i}')
        fft_radix3_manual_fxp.append(complex(stage1_radix3_fxp.output_sample_r, stage1_radix3_fxp.output_sample_i))

fft_radix3_manual_fxp = np.array(fft_radix3_manual_fxp)

i = 8, real = 45.0, imag = 0.0
i = 9, real = -4.5, imag = 2.597900390625
i = 10, real = -4.5, imag = -2.597900390625
i = 11, real = -0.001220703125, imag = 0.00048828125
i = 12, real = 0.0, imag = 0.0009765625
i = 13, real = -13.49853515625, imag = 7.791748046875
i = 14, real = -0.001220703125, imag = -0.000732421875
i = 15, real = -13.49755859375, imag = -7.79345703125
i = 16, real = -0.000732421875, imag = 0.00048828125


## Radix-5 SDF Butterfly FXP 

In [36]:
class radix5_PreAdder_fxp:
    def __init__(self):
        self.input_0_r = Fxp(0.0).like(DATA)
        self.input_0_i = Fxp(0.0).like(DATA)
        self.input_1_r = Fxp(0.0).like(DATA)
        self.input_1_i = Fxp(0.0).like(DATA)
        self.input_2_r = Fxp(0.0).like(DATA)
        self.input_2_i = Fxp(0.0).like(DATA)
        self.input_3_r = Fxp(0.0).like(DATA)
        self.input_3_i = Fxp(0.0).like(DATA)
        self.input_4_r = Fxp(0.0).like(DATA)
        self.input_4_i = Fxp(0.0).like(DATA)

        self.output_0_r = Fxp(0.0).like(DATA)
        self.output_0_i = Fxp(0.0).like(DATA)
        self.output_1_r = Fxp(0.0).like(DATA)
        self.output_1_i = Fxp(0.0).like(DATA)
        self.output_2_r = Fxp(0.0).like(DATA)
        self.output_2_i = Fxp(0.0).like(DATA)
        self.output_3_r = Fxp(0.0).like(DATA)
        self.output_3_i = Fxp(0.0).like(DATA)
        self.output_4_r = Fxp(0.0).like(DATA)
        self.output_4_i = Fxp(0.0).like(DATA)

        # multiplication with -1/4 is implemented below in calculate() function

        self.mult_const_0_r = Fxp(1/2 * (np.cos(2*np.pi/5) - np.cos(4*np.pi/5))).like(DATA)

        self.mult_const_1_cmplx = (-1j*0.363)
        # self.mult_const_0_r = Fxp(self.mult_const_0_cmplx.real).like(DATA)
        self.mult_const_1_i = Fxp(np.sin(4*np.pi/5) - np.sin(2*np.pi/5)).like(DATA)

        self.mult_const_2_cmplx = 1j*1.539
        # self.mult_const_1_r = Fxp(self.mult_const_1_cmplx.real).like(DATA)
        self.mult_const_2_i = Fxp(np.sin(4*np.pi/5) + np.sin(2*np.pi/5)).like(DATA)

        self.mult_const_3_cmplx = (-1j*0.588)
        # self.mult_const_2_r = Fxp(self.mult_const_2_cmplx.real).like(DATA)
        self.mult_const_3_i = Fxp(-np.sin(4*np.pi/5)).like(DATA)



    def calculate(self):
        # First subscript number is signal index, second number is pipeline stage index
        tmp_0_0_r = self.input_0_r
        tmp_0_0_i = self.input_0_i

        tmp_1_0_r = self.input_1_r + self.input_4_r
        tmp_1_0_i = self.input_1_i + self.input_4_i

        tmp_2_0_r = self.input_2_r + self.input_3_r
        tmp_2_0_i = self.input_2_i + self.input_3_i

        tmp_3_0_r = self.input_1_r - self.input_4_r
        tmp_3_0_i = self.input_1_i - self.input_4_i

        tmp_4_0_r = self.input_2_r - self.input_3_r
        tmp_4_0_i = self.input_2_i - self.input_3_i
        ###### First pipeline level ^
        tmp_0_1_r = tmp_0_0_r
        tmp_0_1_i = tmp_0_0_i

        tmp_1_1_r = tmp_1_0_r + tmp_2_0_r
        tmp_1_1_i = tmp_1_0_i + tmp_2_0_i

        tmp_2_1_r = tmp_1_0_r - tmp_2_0_r
        tmp_2_1_i = tmp_1_0_i - tmp_2_0_i

        tmp_3_1_r = tmp_3_0_r
        tmp_3_1_i = tmp_3_0_i

        tmp_4_1_r = tmp_4_0_r
        tmp_4_1_i = tmp_4_0_i

        tmp_5_1_r = tmp_3_0_r + tmp_4_0_r # dodatna grana izmedju
        tmp_5_1_i = tmp_3_0_i + tmp_4_0_i # dodatna grana izmedju
        ###### Second pipeline level ^
        tmp_0_2_r = tmp_0_1_r + tmp_1_1_r
        tmp_0_2_i = tmp_0_1_i + tmp_1_1_i

        tmp_1_2_r = tmp_0_1_r + tmp_1_1_r * (-0.25)
        tmp_1_2_i = tmp_0_1_i + tmp_1_1_i * (-0.25)
        # print("tmp_1_2 = ", tmp_1_2_r, " +j ", tmp_1_2_i)

        tmp_2_2_r = tmp_2_1_r * self.mult_const_0_r
        tmp_2_2_i = tmp_2_1_i * self.mult_const_0_r
        # print("tmp_2_2 = ", tmp_2_2_r, " +j ", tmp_2_2_i)

        # Complex multiplications, but with imaginary factors
        tmp_3_2_r = (-1) * tmp_3_1_i * self.mult_const_1_i
        tmp_3_2_i = tmp_3_1_r * self.mult_const_1_i
        # print("tmp_3_2 = ", tmp_3_2_r, " +j ", tmp_3_2_i)

        tmp_4_2_r = (-1) * tmp_4_1_i * self.mult_const_2_i
        tmp_4_2_i = tmp_4_1_r * self.mult_const_2_i
        # print("tmp_4_2 = ", tmp_4_2_r, " +j ", tmp_4_2_i)

        tmp_5_2_r = (-1) * tmp_5_1_i * self.mult_const_3_i
        tmp_5_2_i = tmp_5_1_r * self.mult_const_3_i
        # print("tmp_5_2 = ", tmp_5_2_r, " +j ", tmp_5_2_i)
        ###### Third pipeline level ^
        tmp_0_3_r = tmp_0_2_r
        tmp_0_3_i = tmp_0_2_i

        tmp_1_3_r = tmp_1_2_r + tmp_2_2_r
        tmp_1_3_i = tmp_1_2_i + tmp_2_2_i

        tmp_2_3_r = tmp_1_2_r - tmp_2_2_r
        tmp_2_3_i = tmp_1_2_i - tmp_2_2_i

        tmp_3_3_r = tmp_3_2_r + tmp_5_2_r
        tmp_3_3_i = tmp_3_2_i + tmp_5_2_i

        tmp_4_3_r = tmp_4_2_r + tmp_5_2_r
        tmp_4_3_i = tmp_4_2_i + tmp_5_2_i
        ###### Fourth pipeline level ^
        self.output_0_r = tmp_0_3_r
        self.output_0_i = tmp_0_3_i

        self.output_1_r = tmp_1_3_r + tmp_3_3_r
        self.output_1_i = tmp_1_3_i + tmp_3_3_i

        self.output_2_r = tmp_2_3_r + tmp_4_3_r
        self.output_2_i = tmp_2_3_i + tmp_4_3_i

        self.output_4_r = tmp_1_3_r - tmp_3_3_r
        self.output_4_i = tmp_1_3_i - tmp_3_3_i

        self.output_3_r = tmp_2_3_r - tmp_4_3_r
        self.output_3_i = tmp_2_3_i - tmp_4_3_i

class radix5_Rotator_fxp:
    cnt = 0
    def __init__(self, stage_index, size):
        self.input_r = Fxp(0.0).like(DATA)
        self.input_i = Fxp(0.0).like(DATA)

        self.output_r = Fxp(0.0).like(DATA)
        self.output_i = Fxp(0.0).like(DATA)
        self.stage_index = stage_index
        
        np_rom = np.zeros((size,2))
        np_rom[:,0].fill(1.0)
        self.twiddleROM = Fxp(np_rom).like(DATA)
        # self.twiddleROM = (np.ones(size)).astype(complex)
        # self.four_fifths_len = 5**(num_of_stages-stage_index) - (5**(num_of_stages-stage_index)//5)
        self.four_fifths_len = size - (size//5)
        N = size
        if (self.four_fifths_len > 4):
            for i in range(self.four_fifths_len):
                if (i < self.four_fifths_len/4):
                    k = i * 5**(stage_index)
                elif ((i >= self.four_fifths_len/4) and (i < self.four_fifths_len/2)):
                    # UPITNO
                    k = 2*(i-self.four_fifths_len//4) * 5**(stage_index)
                elif ((i >= self.four_fifths_len/2) and (i < 3*self.four_fifths_len/4)):
                    # UPITNO
                    k = 3*(i-2*self.four_fifths_len//4) * 5**(stage_index)
                else:
                    k = 4*(i-3*self.four_fifths_len//4) * 5**(stage_index)
                # print("k = ", k)
                twiddle_factor = np.exp(-1j*2*np.pi*k/N)
                self.twiddleROM[i+(len(self.twiddleROM) - self.four_fifths_len), 0] = twiddle_factor.real
                self.twiddleROM[i+(len(self.twiddleROM) - self.four_fifths_len), 1] = twiddle_factor.imag
        
        # print(f"STAGE {stage_index}, twiddle = {self.twiddleROM}")

        # print(self.twiddleROM)

    def rotate(self, fifo_full_flag):
        x_r = Fxp(0.0).like(DATA)
        x_i = Fxp(0.0).like(DATA)
        y_r = Fxp(0.0).like(DATA)
        y_i = Fxp(0.0).like(DATA)
        z_r = Fxp(0.0).like(DATA)
        z_i = Fxp(0.0).like(DATA)
        if (fifo_full_flag): # dozvola da brojac vrti i cita redom twiddle faktore iz memorije
            # print(f"STAGE {self.stage_index}, FIFO FULL")
            x_r(self.input_r)
            x_i(self.input_i)
            y_r(self.twiddleROM[self.cnt, 0])
            y_i(self.twiddleROM[self.cnt, 1])
            # self.output = self.input * self.twiddleROM[self.cnt]
            z_r.set_val(x_r*y_r - x_i*y_i)
            z_i.set_val(x_r*y_i + x_i*y_r)

            self.output_r.set_val(z_r)
            self.output_i.set_val(z_i)
            # print(f'STAGE {self.stage_index}, curr twiddle = {self.twiddleROM[self.cnt]}')
            if (self.cnt == len(self.twiddleROM)-1):
                self.cnt = 0
            else:
                self.cnt += 1

In [37]:
class radix5_SDF_fxp_stage:
    input_sample_r = Fxp(0.0).like(DATA)
    input_sample_i = Fxp(0.0).like(DATA)
    output_sample_r = Fxp(0.0).like(DATA)
    output_sample_i = Fxp(0.0).like(DATA)
    op_cnt = 0 # operation counter (counting how many add/subb operations are done)
    def __init__(self, stage_index, size):
        self.stage_index = stage_index
        self.num_of_samples = size
        self.fifo_0 = Fifo_fxp(size//5)
        self.fifo_1 = Fifo_fxp(size//5)
        self.fifo_2 = Fifo_fxp(size//5)
        self.fifo_3 = Fifo_fxp(size//5)
        self.pre_adder = radix5_PreAdder_fxp()
        self.rotator = radix5_Rotator_fxp(stage_index=stage_index, size=size)

    def isFifoFull_3(self):
        return self.fifo_3.is_full()
    
    def calculate(self):
        fifo_0_out_reg = self.fifo_0.get_output()
        fifo_1_out_reg = self.fifo_1.get_output()
        fifo_2_out_reg = self.fifo_2.get_output()
        fifo_3_out_reg = self.fifo_3.get_output()
        self.pre_adder.input_0_r.set_val(fifo_0_out_reg[0])
        self.pre_adder.input_0_i.set_val(fifo_0_out_reg[1])
        self.pre_adder.input_1_r.set_val(fifo_1_out_reg[0])
        self.pre_adder.input_1_i.set_val(fifo_1_out_reg[1])
        self.pre_adder.input_2_r.set_val(fifo_2_out_reg[0])
        self.pre_adder.input_2_i.set_val(fifo_2_out_reg[1])
        self.pre_adder.input_3_r.set_val(fifo_3_out_reg[0])
        self.pre_adder.input_3_i.set_val(fifo_3_out_reg[1])
        self.pre_adder.input_4_r.set_val(self.input_sample_r)
        self.pre_adder.input_4_i.set_val(self.input_sample_i)
        self.pre_adder.calculate()
        
        if (self.op_cnt < (self.num_of_samples//5)): ## other half of the input stream is comming
            self.output_sample_r.set_val(fifo_0_out_reg[0])
            self.output_sample_i.set_val(fifo_0_out_reg[1])
            self.fifo_0.shift(self.input_sample_r, self.input_sample_i)
        elif ((self.op_cnt >= (self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*2/5))):
            self.output_sample_r.set_val(fifo_1_out_reg[0])
            self.output_sample_i.set_val(fifo_1_out_reg[1])
            self.fifo_1.shift(self.input_sample_r, self.input_sample_i)
        elif ((self.op_cnt >= (2*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*3/5))):
            self.output_sample_r.set_val(fifo_2_out_reg[0])
            self.output_sample_i.set_val(fifo_2_out_reg[1])
            self.fifo_2.shift(self.input_sample_r, self.input_sample_i)
        elif ((self.op_cnt >= (3*self.num_of_samples//5)) and (self.op_cnt < (self.num_of_samples*4/5))):
            self.output_sample_r.set_val(fifo_3_out_reg[0])
            self.output_sample_i.set_val(fifo_3_out_reg[1])
            self.fifo_3.shift(self.input_sample_r, self.input_sample_i)
        else:
            self.output_sample_r.set_val(self.pre_adder.output_0_r)
            self.output_sample_i.set_val(self.pre_adder.output_0_i)
            self.fifo_0.shift(self.pre_adder.output_1_r, self.pre_adder.output_1_i)
            self.fifo_1.shift(self.pre_adder.output_2_r, self.pre_adder.output_2_i)
            self.fifo_2.shift(self.pre_adder.output_3_r, self.pre_adder.output_3_i)
            self.fifo_3.shift(self.pre_adder.output_4_r, self.pre_adder.output_4_i)

        self.rotator.input_r.set_val(self.output_sample_r)
        self.rotator.input_i.set_val(self.output_sample_i)
        self.rotator.rotate(self.isFifoFull_3())
        self.output_sample_r.set_val(self.rotator.output_r)
        self.output_sample_i.set_val(self.rotator.output_i)

        # print(f'stage_{self.stage_index} : {self.output_sample}')

        if (self.op_cnt == self.num_of_samples-1):
            self.op_cnt = 0
        else:
            self.op_cnt += 1

        # print(f'STAGE {self.stage_index}, output = {self.output_sample}')

In [38]:
stage0_radix5_fxp = radix5_SDF_fxp_stage(stage_index=0, size=25)
stage1_radix5_fxp = radix5_SDF_fxp_stage(stage_index=0, size=5)

# input_vector = [1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
input_vector_r = np.append(np.arange(25), np.zeros(24))
input_vector_r = Fxp(input_vector_r).like(DATA)
input_vector_i = Fxp(np.zeros(49)).like(DATA)

fft_radix5_manual_fxp = []
for i in range(len(input_vector_r)):
    stage0_radix5_fxp.input_sample_r.set_val(input_vector_r[i])
    stage0_radix5_fxp.input_sample_i.set_val(input_vector_i[i])
    stage0_radix5_fxp.calculate()
    stage1_radix5_fxp.input_sample_r.set_val(stage0_radix5_fxp.output_sample_r)
    stage1_radix5_fxp.input_sample_i.set_val(stage0_radix5_fxp.output_sample_i)
    stage1_radix5_fxp.calculate()
    # print(stage1_radix5_fxp.output_sample)
    if i >= 24:
        print(f'i = {i}, real = {stage1_radix5_fxp.output_sample_r}, imag = {stage1_radix5_fxp.output_sample_i}')
        fft_radix5_manual_fxp.append(complex(stage1_radix5_fxp.output_sample_r, stage1_radix5_fxp.output_sample_i))

fft_radix5_manual_fxp = np.array(fft_radix5_manual_fxp)

i = 24, real = 300.0, imag = 0.0
i = 25, real = -12.5, imag = 17.198486328125
i = 26, real = -12.5, imag = 4.058837890625
i = 27, real = -12.5, imag = -4.058837890625
i = 28, real = -12.5, imag = -17.198486328125
i = 29, real = -12.516357421875, imag = 98.905029296875
i = 30, real = -12.495361328125, imag = 13.305908203125
i = 31, real = -12.50048828125, imag = 2.384765625
i = 32, real = -12.492431640625, imag = -5.879638671875
i = 33, real = -12.495849609375, imag = -22.72314453125
i = 34, real = -12.509521484375, imag = 48.6728515625
i = 35, real = -12.504150390625, imag = 10.32958984375
i = 36, real = -12.499267578125, imag = 0.77880859375
i = 37, real = -12.4931640625, imag = -7.932861328125
i = 38, real = -12.494384765625, imag = -31.553466796875
i = 39, real = -12.492431640625, imag = 31.56396484375
i = 40, real = -12.4970703125, imag = 7.9208984375
i = 41, real = -12.499267578125, imag = -0.787353515625
i = 42, real = -12.504150390625, imag = -10.33447265625
i = 43, real = -12.5

## SQNR evaluation

In [39]:
import numpy as np
from scipy.signal import chirp
from scipy.fftpack import fft

# Fixed-point conversion parameters
N_WORD = 16  # Total number of bits
N_FRAC = 12  # Fractional bits (Q-format: Q(N_WORD-N_FRAC).N_FRAC)

# FFT size
N = 8  

def float_to_fixed(x, N_WORD, N_FRAC):
    """Convert floating-point values to fixed-point representation."""
    scale = 2**N_FRAC
    x_fixed = np.round(x * scale).astype(np.int16)  # Convert to int16
    return x_fixed

def fixed_to_float(x_fixed, N_FRAC):
    """Convert fixed-point values back to floating-point."""
    return x_fixed / (2**N_FRAC)

def generate_sine_wave(N, k, A=1.0):
    """ Generate a sinusoidal test signal """
    n = np.arange(N)
    x = A * np.sin(2 * np.pi * k * n / N)
    return x

def generate_multitone(N, freqs, A=1.0):
    """ Generate a multi-tone test signal """
    n = np.arange(N)
    x = sum(A * np.sin(2 * np.pi * f * n / N) for f in freqs)
    x /= max(abs(x))  # Normalize to avoid overflow
    return x

def generate_white_noise(N, A=1.0):
    """ Generate a white noise signal """
    x = A * (2 * np.random.rand(N) - 1)  # Uniform noise in range [-A, A]
    return x

def generate_chirp(N, f0, f1, A=1.0):
    """ Generate a linear chirp signal """
    t = np.linspace(0, 1, N)
    x = A * chirp(t, f0=f0, f1=f1, t1=1, method='linear')
    return x

def compute_sqnr(x_float, x_fixed):
    """ Compute SQNR between floating-point and fixed-point FFT results """
    P_signal = np.mean(np.abs(x_float) ** 2)
    P_noise = np.mean(np.abs(x_float - x_fixed) ** 2)
    
    SQNR = 10 * np.log10(P_signal / P_noise) if P_noise > 0 else np.inf
    return SQNR

# Select test signal
test_signal = 'sine'  # Options: 'sine', 'multi', 'noise', 'chirp'

if test_signal == 'sine':
    x = generate_sine_wave(N, k=5, A=0.9)
elif test_signal == 'multi':
    x = generate_multitone(N, [3, 7, 15], A=0.9)
elif test_signal == 'noise':
    x = generate_white_noise(N, A=0.9)
elif test_signal == 'chirp':
    x = generate_chirp(N, f0=1, f1=30, A=0.9)

# Convert to fixed-point
x_fixed = float_to_fixed(x, N_WORD, N_FRAC)

# Compute FFT (floating-point)
X_float = fft(x)

# Compute FFT (fixed-point)
X_fixed = fft(fixed_to_float(x_fixed, N_FRAC))

# Compute SQNR
sqnr_value = compute_sqnr(X_float, X_fixed)
print(f"SQNR for {test_signal} signal: {sqnr_value:.2f} dB")


SQNR for sine signal: 78.70 dB


In [40]:
# Compute SQNR
sqnr_value = compute_sqnr(fft_numpy, fft_manual_fxp)
print(f"SQNR for signal: {sqnr_value:.2f} dB")

SQNR for signal: 90.48 dB


In [41]:
np.exp(-1j*2*np.pi*2/3)

np.complex128(-0.5000000000000004+0.8660254037844384j)

In [42]:
np.exp(-1j*2*np.pi*1/3)

np.complex128(-0.4999999999999998-0.8660254037844387j)

In [43]:
(-1j*np.sqrt(3)/2)

-0.8660254037844386j

# Twiddle Factor Generator

